# Trefle API exploration

This notebook exercises a few Trefle API requests to understand the response structure, available fields, and common gaps in the data.

In [ ]:
import json
import os
import urllib.parse
import urllib.request
from pathlib import Path

In [ ]:
# Load the Trefle token from the workspace .env file.
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.env').exists():
            return candidate
    return start

repo_root = find_repo_root()
env_path = repo_root / '.env'
token = None
if env_path.exists():
    for line in env_path.read_text().splitlines():
        if line.startswith('TREFLE_API_KEY='):
            token = line.split('=', 1)[1].strip()
            break

if not token:
    token = os.getenv('TREFLE_API_KEY') or os.getenv('TREFLE_TOKEN')

if not token:
    raise RuntimeError('No Trefle token found. Check your .env file or environment variables.')

def fetch_trefle(path: str, params: dict | None = None):
    base_url = 'https://trefle.io'
    query = {'token': token}
    if params:
        query.update(params)
    url = base_url + path + '?' + urllib.parse.urlencode(query)
    with urllib.request.urlopen(url, timeout=20) as response:
        return json.load(response)

print('Token loaded successfully.')

In [ ]:
# 1) List plants from the API.
plants_response = fetch_trefle('/api/v1/plants', {'page': 1, 'limit': 5})
print('Top-level keys:', list(plants_response.keys()))
print('Number of records returned:', len(plants_response.get('data', [])))
print('First record sample:')
print(json.dumps(plants_response['data'][0], indent=2)[:2500])

In [ ]:
# 2) Search for a term like 'oak' using the documented plants/search endpoint.
oak_response = fetch_trefle('/api/v1/plants/search', {'q': 'oak', 'limit': 5})
oak_items = oak_response.get('data', [])
print('Search result count:', len(oak_items))
for item in oak_items[:5]:
    print(item.get('common_name'), '->', item.get('scientific_name'))

In [ ]:
# 3) Fetch a specific plant by slug/identifier and inspect its nested fields.
sample_slug = 'quercus-rotundifolia'
species_response = fetch_trefle(f'/api/v1/plants/{sample_slug}')
species_data = species_response.get('data', {})
print('Specific record keys:', sorted(species_data.keys()))
print(json.dumps(species_data, indent=2)[:3500])

In [ ]:
# 4) Compare field availability across several results.
records = plants_response.get('data', [])
field_names = sorted({field for record in records for field in record.keys()})
print('Fields present across the first page of results:')
for field in field_names:
    print('-', field)

# Print a quick view of which values are often missing.
for field in ['family_common_name', 'image_url', 'author', 'bibliography']:
    count = sum(1 for record in records if record.get(field))
    print(f'{field}: {count}/{len(records)} filled')

In [ ]:
# 5) Search for additional terms using the documented search endpoint.
search_terms = ['blueberry', 'annona', 'mangosteen']
search_responses = {}
for term in search_terms:
    search_responses[term] = fetch_trefle('/api/v1/plants/search', {'q': term, 'limit': 5})
    print(f'Query: {term}')
    print('result count:', len(search_responses[term].get('data', [])))
    for item in search_responses[term].get('data', [])[:5]:
        print(' -', item.get('common_name'), '->', item.get('scientific_name'))
    print()

# 6) Try a few documented filters and ordering options.
common_name_filter = fetch_trefle('/api/v1/plants', {'filter[common_name]': 'beach strawberry', 'limit': 5})
print('Common-name filter results:')
for item in common_name_filter.get('data', [])[:5]:
    print(' -', item.get('common_name'), '->', item.get('scientific_name'))

edible_part_filter = fetch_trefle('/api/v1/species', {'filter[edible_part]': 'roots,leaves', 'limit': 5})
print('\nEdible-part filter results:')
for item in edible_part_filter.get('data', [])[:5]:
    print(' -', item.get('common_name'), '->', item.get('scientific_name'))

ordered_response = fetch_trefle('/api/v1/plants', {'order[year]': 'asc', 'order[scientific_name]': 'desc', 'limit': 5})
print('\nOrdered results:')
for item in ordered_response.get('data', [])[:5]:
    print(' -', item.get('common_name'), '->', item.get('scientific_name'))

In [ ]:
# 6) Save a small sample of the raw JSON to disk for later inspection.
output_path = repo_root / 'data' / 'raw' / 'trefle_api_sample.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps({
    'plants_page_1': plants_response,
    'oak_search': oak_response,
    'species_detail': species_response,
    'searches': {term: search_responses[term] for term in search_terms}
}, indent=2))
print(f'Saved sample data to {output_path}')

## Additional Trefle API feature tests

The next cells exercise the other documented Trefle capabilities: filter_not, edible-part filters, range filtering, sorting, pagination, and scientific-name search.

In [ ]:
# 7) Test filter_not and edible-part filters.
edible_not_null = fetch_trefle('/api/v1/plants', {'filter_not[edible_part]': 'null', 'limit': 5})
print('filter_not[edible_part]=null results:')
for item in edible_not_null.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

edible_part_filter = fetch_trefle('/api/v1/plants', {'filter[edible_part]': 'roots', 'limit': 5})
print('\nfilter[edible_part]=roots results:')
for item in edible_part_filter.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

In [ ]:
# 8) Test range filtering and ordering.
height_range = fetch_trefle('/api/v1/species', {'range[maximum_height_cm]': '5,20', 'limit': 5})
print('range[maximum_height_cm]=5,20 results:')
print('Returned records:', len(height_range.get('data', [])))
for item in height_range.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

# Use a populated range field to demonstrate the filter syntax.
year_range = fetch_trefle('/api/v1/plants', {'range[year]': '1800,2024', 'limit': 5})
print('\nrange[year]=1800,2024 results:')
for item in year_range.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

ordered_response = fetch_trefle('/api/v1/plants', {'order[year]': 'asc', 'order[scientific_name]': 'desc', 'limit': 5})
print('\nOrdered results:')
for item in ordered_response.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

In [ ]:
# 9) Test pagination and scientific-name search.
page_two = fetch_trefle('/api/v1/plants', {'page': 2, 'limit': 3})
print('Page 2 results:')
for item in page_two.get('data', [])[:3]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

scientific_name_search = fetch_trefle('/api/v1/plants/search', {'q': 'Plinia cauliflora', 'limit': 5})
print('\nScientific-name search results:')
for item in scientific_name_search.get('data', [])[:5]:
    print(' -', item.get('common_name') or item.get('scientific_name'))

In [ ]:
# Save the new feature-test outputs alongside the earlier sample.
output_path = repo_root / 'data' / 'raw' / 'trefle_api_sample.json'
output_path.write_text(json.dumps({
    'plants_page_1': plants_response,
    'oak_search': oak_response,
    'species_detail': species_response,
    'searches': {term: search_responses[term] for term in search_terms},
    'feature_tests': {
        'filter_not_edible_part': edible_not_null,
        'edible_part_filter': edible_part_filter,
        'height_range': height_range,
        'ordered_results': ordered_response,
        'page_two': page_two,
        'scientific_name_search': scientific_name_search,
    }
}, indent=2))
print(f'Updated sample data at {output_path}')

## Data Quality Notes

- The search example for "annona" shows that some names are likely incorrect or conflated. The result "Custard-apple -> Annona squamosa" is problematic because Annona squamosa is commonly known as sugar apple, while custard apple is more commonly associated with Annona reticulata (often called bullock's heart). This suggests that some Trefle entries may be mislabeled or merged incorrectly.
- Trefle is partly community sourced, so naming inconsistencies, synonyms, and human error are expected in the data.
- The saved sample JSON also shows that the data is not always complete. In the first-page plants records, fields such as `family_common_name` are often null, and the output contains many nested attributes with null values for optional characteristics such as precipitation, temperature, and edible parts.
- Some records are also sparse in a more visible way: the saved JSON includes entries where `common_name` is null, which makes search and display behavior less reliable.
- Overall, the data is useful for exploration and quick lookups, but it should be cleaned and validated before using it for analysis, ranking, or modeling.